# Gold Layer — Monthly Revenue

Aggregates total revenue per month from `silver.orders` joined with `silver.order_items`. Excludes canceled and unavailable orders. Output is one row per `year_month` (format `yyyy-MM`).

## Design Decisions

* **Revenue defined as `SUM(price + freight_value)` per row** — I treated each order_items row as a distinct unit with its own freight allocation. An alternative interpretation — that freight is line-level repeated per unit — would call for pre-aggregation per (order, product) line. I chose the per-row interpretation for v1.
* **Status filter on orders** — excludes `canceled` and `unavailable` orders, which represent customer drop-outs rather than realized revenue.
* **Filter before join** — applied the status filter to orders before joining to order_items, reducing the join size.

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'
quarantine_schema = 'quarantine'

# Monthly Revenue

Revenue calculation: per-row sum of price and freight_value. I treated each order_items row as a distinct unit with its own freight allocation. An alternative interpretation — that freight is line-level repeated per unit — would call for pre-aggregation; I chose the per-row interpretation for v1.

In [0]:
silver_orders_df = spark.read.table(f'{catalog_name}.{silver_schema}.orders')\
    .filter(~col('order_status').isin(['canceled', 'unavailable']))\
        .select('order_id','order_purchase_timestamp')

silver_orders_items_df = spark.read.table(f'{catalog_name}.{silver_schema}.order_items').select('order_id','price','freight_value')

joined_df = silver_orders_df.join(silver_orders_items_df, on='order_id', how='inner')\
    .withColumn('year_month',date_format(col('order_purchase_timestamp'),'yyyy-MM'))\
        .withColumn('total_price',col('price')+col('freight_value'))

monthly_revenue_df = joined_df.groupBy('year_month').agg(sum('total_price').alias('total_revenue')).orderBy(col('year_month').desc())

In [0]:
check_not_null(monthly_revenue_df, ['year_month', 'total_revenue'])
check_unique(monthly_revenue_df, ['year_month'])
check_value_range(monthly_revenue_df, [
    {'column': 'total_revenue', 'min_value': 0, 'inclusive': True}
])

check_not_null: PASSED
check_unique: PASSED
check_value_range: PASSED


In [0]:
monthly_revenue_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{gold_schema}.monthly_revenue')